# 🧠 Splotowe sieci neuronowe (CNN) - cz.2

Warto przeczytać: https://www.learnpytorch.io/03_pytorch_computer_vision/

Na poprzednich zajęciach poznaliśmy warstwy tworzące **splotową sieć neuronową** i przypomnieliśmy sobie jak tworzyć modele jako klasy  dziedziczące po `nn.Module`.

Dziś wytrenujemy splotową sieć neuronową do binarnej klasyfikacji obrazu i poznamy dodatkowe techniki stosowane w sieciach neuronowych, m.in. do regularyzacji modeli.

## Pytania kontrolne

1. Opisz budowę splotowej sieci neuronowej. Wyjaśnij, do czego służą jej poszczególne warstwy.
2. Na czym polega regularyzacja modeli?
3. Jakie znasz metody regularyzacji stosowane w sieciach neuronowych?

 # ODPOWIEDZI
 1. Splotowa sieć neuronowa (CNN) służy głównie do analizy obrazów. Składa się z warstw Conv2D, które wykrywają cechy obrazu (np. krawędzie), funkcji aktywacji (np. Tanh/ReLU), które dodają nieliniowość, oraz MaxPooling, który zmniejsza rozmiar danych. Na końcu znajdują się warstwy Linear (gęste) odpowiedzialne za klasyfikację. Dodatkowo można stosować BatchNorm i Dropout, aby poprawić działanie modelu.
2. Regularyzacja polega na ograniczaniu przeuczenia (overfittingu), czyli sytuacji, gdy model dobrze działa na danych treningowych, ale słabo na nowych. Jej celem jest poprawa zdolności generalizacji modelu.
3. Najczęściej stosowane metody to: L2 (weight decay), Dropout, Batch Normalization, Early Stopping, Data Augmentation.

## Kod z poprzednich ćwiczeń

In [4]:
from torchvision import datasets, transforms
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset

import matplotlib.pyplot as plt
from datasets import load_dataset


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

C:\Users\Patryk\PycharmProjects\LAB04\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [5]:
# wczytujemy CIFAR-10 z HuggingFace (w wypadku gdy torchvision rzuca błąd 503) ---
ds = load_dataset("cifar10")

to_tensor = transforms.ToTensor()

train_imgs = torch.stack([
    to_tensor(img["img"]) for img in ds["train"]
])

mean = train_imgs.mean(dim=(0, 2, 3))
std = train_imgs.std(dim=(0, 2, 3))

print("mean:", mean)
print("std:", std)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# PyTorch Dataset wrapper dla zbioru danych z HF
class HFCIFAR(Dataset):
    def __init__(self, hf_split, transform=None):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = self.data[idx]["img"]
        label = self.data[idx]["label"]

        if self.transform:
            img = self.transform(img)

        return img, label

train_ds = HFCIFAR(ds["train"], transform=transform)
val_ds = HFCIFAR(ds["test"], transform=transform)

# Filtrowanie klas
label_map = {0: 0, 2: 1}

class FilteredDataset(Dataset):
    def __init__(self, dataset, label_map):
        self.dataset = dataset
        self.label_map = label_map
        self.indices = [
            i for i in range(len(dataset))
            if dataset[i][1] in label_map
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.dataset[self.indices[idx]]
        return x, self.label_map[y]

train_data = FilteredDataset(train_ds, label_map)
val_data = FilteredDataset(val_ds, label_map)

mean: tensor([0.4914, 0.4822, 0.4465])
std: tensor([0.2470, 0.2435, 0.2616])


In [6]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 8, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(8 * 8 * 8, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = F.max_pool2d(torch.tanh(self.conv1(x)), 2)
        out = F.max_pool2d(torch.tanh(self.conv2(out)), 2)
        out = out.view(-1, 8 * 8 * 8)
        out = torch.tanh(self.fc1(out))
        out = self.fc2(out)
        return out

### ✅ Zadanie 1
Uzupełnij poniższe komórki, aby wytrenować splotową sieć neuronową do zadania klasyfikacji binarnej.

Przyjmij learning rate o wartości `0.01` i batch size `64`. Trenuj przez `100` epok.


*Pamiętaj o użyciu GPU

In [7]:
def training_loop(n_epochs, optimizer, model, loss_fn, train_loader, device):
    print(f'Training on: {device}')
    for epoch in range(1, n_epochs + 1):
        loss_train = 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(device=device)
            labels = labels.to(device=device)
            outputs = model(imgs)
            loss = loss_fn(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_train += loss.item()

        if epoch == 1 or epoch % 10 == 0:
            print(f"Epoch {epoch}, Training loss {loss_train}")

In [8]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "brak gpu")

True
1
NVIDIA GeForce RTX 3070 Ti


In [9]:
import torch
print(torch.__version__)
print(torch.version.cuda)

2.11.0+cu128
12.8


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

model = Net().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

loss_fn = nn.CrossEntropyLoss()

training_loop(
    n_epochs=100,
    optimizer=optimizer,
    model=model,
    loss_fn=loss_fn,
    train_loader=train_loader,
    device=device
)

Training on: cuda
Epoch 1, Training loss 70.7701740860939
Epoch 10, Training loss 51.93173699080944
Epoch 20, Training loss 53.724399492144585
Epoch 30, Training loss 48.39754539728165
Epoch 40, Training loss 50.66871991753578
Epoch 50, Training loss 46.67576062679291
Epoch 60, Training loss 46.205645814538
Epoch 70, Training loss 43.776873990893364
Epoch 80, Training loss 44.952959299087524
Epoch 90, Training loss 47.35309541225433
Epoch 100, Training loss 44.62485855817795


### ✅ Zadanie 2

Zaimplementuj funkcję `validate`, która zmierzy **dokładność, precyzję, czułość, F1** wytrenowanego modelu na dwóch zbiorach - uczącym i walidacyjnym.

*Pamiętaj o użyciu GPU

Porównaj wyniki z wynikami z **laboratorium nr 6** (sieć gęsta).

In [12]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score


val_loader = torch.utils.data.DataLoader(
    val_data,
    batch_size=64,
    shuffle=False
)


def validate(model, loader, device):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for imgs, labels in loader:

            imgs = imgs.to(device=device)
            labels = labels.to(device=device)

            outputs = model(imgs)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return acc, prec, rec, f1


train_metrics = validate(
    model,
    train_loader,
    device
)

val_metrics = validate(
    model,
    val_loader,
    device
)

print("train:")
print(f"accuracy: {train_metrics[0]:.4f}")
print(f"precision: {train_metrics[1]:.4f}")
print(f"recall: {train_metrics[2]:.4f}")
print(f"f1: {train_metrics[3]:.4f}")

print("\nvalidation:")
print(f"accuracy: {val_metrics[0]:.4f}")
print(f"precision: {val_metrics[1]:.4f}")
print(f"recall: {val_metrics[2]:.4f}")
print(f"f1: {val_metrics[3]:.4f}")

train:
accuracy: 0.8879
precision: 0.8930
recall: 0.8814
f1: 0.8872

validation:
accuracy: 0.8700
precision: 0.8783
recall: 0.8590
f1: 0.8686


# Porównanie z LAB06
W laboratorium 6 wykorzystano sieć gęstą (fully connected), która osiągnęła dokładność około 52\%, co oznaczało wynik bliski losowemu zgadywaniu. W laboratorium 8 zastosowano sieć splotową CNN, która uzyskała znacznie lepsze wyniki na zbiorze walidacyjnym: accuracy = 87.00\%, precision = 87.83\%, recall = 85.90\% oraz F1-score = 86.86\%. Oznacza to poprawę skuteczności klasyfikacji o około 35 punktów procentowych względem laboratorium 6. Lepsze wyniki wynikają z zastosowania warstw splotowych, które potrafią wykrywać charakterystyczne cechy obrazu i zachowują informacje przestrzenne.

## Zapis i odczyt modelu

Wytrenowany model (zwłaszcza tak, którego trening trwa długo) warto zapisać, aby móc go użyć później.

Typowo po każdej epoce treningu sprawdza się działanie modelu na zbiorze walidacyjnym (walidacja).
Zapisu modelu (tzw. "checkpoint") typowo dokonuje się, jeśli wartość danej metryki (np. dokładność lub F1 na zbiorze walidacyjnym) jest lepsza niż najlepsza uzyskana dotychczas.

PyTorch pozwala zapisać wagi (parametry) modelu z użyciem `torch.save` oraz tzw. `state_dict` modelu (https://pytorch.org/tutorials/recipes/recipes/what_is_state_dict.html). Innym sposobem zapisu jest zapis całego modelu (z użyciem `pickle` "pod spodem"):
https://pytorch.org/tutorials/recipes/recipes/saving_and_loading_models_for_inference.html

Następnie, do wczytania zapisanego modelu można użyć metody `load_state_dict` (oraz `torch.load`), jeśli zapisywaliśmy tylko `state_dict` lub tylko `torch.load`, jeśli zapisywaliśmy cały model.

### ✅ Zadnie 3
Zapisz wytrenowany model, a następnie wczytaj go ponownie.

In [13]:
# zapis modelu
torch.save(model.state_dict(), "cnn_model.pth")

print("Model zapisany")

Model zapisany


In [14]:
# nowa instancja modelu
loaded_model = Net().to(device)

# wczytanie wag
loaded_model.load_state_dict(
    torch.load("cnn_model.pth", map_location=device)
)

# tryb ewaluacji
loaded_model.eval()

print("Model wczytany poprawnie")

Model wczytany poprawnie


In [15]:
# test pojedynczego batcha
images, labels = next(iter(val_loader))

images = images.to(device)

with torch.no_grad():
    outputs = loaded_model(images)
    preds = torch.argmax(outputs, dim=1)

print(preds[:10])

tensor([0, 0, 0, 1, 1, 1, 0, 0, 1, 0], device='cuda:0')


## Rozbudowa modelu

Możemy "powiększyć" model "na szerokość" (dodać więcej filtrów) lub "na głębokość" (dodać więcej warstw).

### ✅ Zadanie 4

Zmodyfikuj klasę `Net` i stwórz kolejno:
- `NetWidth` - 2x więcej filtrów w warstwach splotowych (niech liczba filtrów będzie argumentem konstruktora)
- `NetDepth` - dodatkowa warstwa splotowa `conv3`

Pamiętaj o zmodyfikowaniu metody `forward`.

In [16]:
import torch
import torch.nn as nn

class NetWidth(nn.Module):

    def __init__(self, filters=32):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=filters,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=filters,
            out_channels=filters // 2,
            kernel_size=3,
            padding=1
        )

        self.act1 = nn.Tanh()
        self.act2 = nn.Tanh()
        self.act3 = nn.Tanh()

        self.pool1 = nn.MaxPool2d(2)
        self.pool2 = nn.MaxPool2d(2)

        self.fc1 = nn.Linear((filters // 2) * 8 * 8, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):

        x = self.conv1(x)
        x = self.act1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.act2(x)
        x = self.pool2(x)

        x = torch.flatten(x, 1)

        x = self.fc1(x)
        x = self.act3(x)

        x = self.fc2(x)

        return x


model_width = NetWidth(filters=32).to(device)

print(model_width)

NetWidth(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (act1): Tanh()
  (act2): Tanh()
  (act3): Tanh()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1024, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=2, bias=True)
)


In [17]:
import torch
import torch.nn as nn

class NetDepth(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=16,
            out_channels=8,
            kernel_size=3,
            padding=1
        )

        self.conv3 = nn.Conv2d(
            in_channels=8,
            out_channels=4,
            kernel_size=3,
            padding=1
        )

        self.act1 = nn.Tanh()
        self.act2 = nn.Tanh()
        self.act3 = nn.Tanh()
        self.act4 = nn.Tanh()

        self.pool1 = nn.MaxPool2d(2)
        self.pool2 = nn.MaxPool2d(2)
        self.pool3 = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(4 * 4 * 4, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):

        x = self.conv1(x)
        x = self.act1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.act2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.act3(x)
        x = self.pool3(x)

        x = torch.flatten(x, 1)

        x = self.fc1(x)
        x = self.act4(x)

        x = self.fc2(x)

        return x


model_depth = NetDepth().to(device)

print(model_depth)

NetDepth(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(8, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (act1): Tanh()
  (act2): Tanh()
  (act3): Tanh()
  (act4): Tanh()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=2, bias=True)
)


## Regularyzacja

**Regularyzacja** to zbiór technik stosowanych w modelach uczenia maszynowego w celu **ograniczenia overfittingu (przeuczenia)**, czyli sytuacji, gdy model bardzo dobrze dopasowuje się do danych treningowych, ale słabo radzi sobie na nowych danych.

---

### 🎯 Po co stosujemy regularyzację?

Bez regularyzacji model może:
- „zapamiętać” dane treningowe zamiast nauczyć się ogólnych zależności  
- być bardzo wrażliwy na szum w danych  
- mieć słabą jakość predykcji na zbiorze testowym  

Regularyzacja pomaga:
- poprawić **generalizację** (działanie na nowych danych)  
- uprościć model  
- ustabilizować proces uczenia  

---

### 🧠 Intuicja

Regularyzacja działa jak „kara za zbyt skomplikowany model”.  
Model jest zachęcany do znajdowania prostszych rozwiązań, które lepiej uogólniają.

---

### 🔧 Najważniejsze metody regularyzacji

#### 1. Regularyzacja L2 (weight decay)
- dodaje karę za duże wartości wag: $\sum w_i^2$
- prowadzi do „mniejszych”, bardziej rozłożonych wag  
- implementowana np. przez `weight_decay` w optymalizatorze  

#### 2. Regularyzacja L1
- dodaje karę: $\sum |w_i|$
- prowadzi do **sparsity** (wiele wag = 0)  
- może działać jak selekcja cech  

#### 3. Dropout
- losowo wyłącza neurony podczas treningu  
- zmusza model do uczenia się redundancji  
- szczególnie popularny w głębokich sieciach neuronowych  

#### 4. Batch Normalization
- normalizuje aktywacje w trakcie treningu  
- stabilizuje i przyspiesza uczenie  
- wprowadza efekt regularyzacyjny (choć nie jest klasyczną regularyzacją)  

#### 5. Early stopping
- przerywa trening, gdy jakość na walidacji przestaje się poprawiać  
- zapobiega dalszemu „przeuczaniu” modelu  

#### 6. Data augmentation
- sztuczne zwiększanie zbioru danych (np. obracanie obrazów)  
- model widzi więcej wariantów danych → lepsza generalizacja  

### 📌 Podsumowanie

Regularyzacja to kluczowy element budowy modeli:
- zapobiega przeuczeniu  
- poprawia jakość na nowych danych  
- pozwala trenować bardziej stabilne i użyteczne modele  

W praktyce często stosuje się **kilka metod jednocześnie** (np. L2 + dropout + data augmentation).

## Regularyzacja L2 (weight decay)

Regularyzacja L2 to technika ograniczająca **overfitting (przeuczenie)** modelu.  
Polega na dodaniu do funkcji straty kary za duże wartości wag — dzięki temu model uczy się prostszych, bardziej ogólnych zależności.

---

### 🔧 Idea matematyczna

Do standardowej funkcji straty dodajemy składnik:

$$
L_{total} = L_{data} + \lambda \sum_i w_i^2
$$

- $L_{data}$ – podstawowa funkcja straty (np. cross-entropy)  
- $lambda$ – siła regularyzacji (`weight_decay`)  
- $w_i$ – wagi modelu  

Im większe wagi, tym większa kara → model unika „przesadnego dopasowania”.

---

### 🧠 Intuicja

- bez regularyzacji: model może dopasować się „za dobrze” do danych treningowych  
- z regularyzacją L2: duże wagi są karane → model jest prostszy i lepiej generalizuje  

---

### ⚙️ Implementacja w PyTorch

#### ✅ Wbudowana (zalecana)

```python
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, weight_decay=1e-4)

In [18]:
def training_loop_l2reg(n_epochs, optimizer, model, loss_fn, train_loader):
    for epoch in range(1, n_epochs + 1):
        loss_train = 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(device=device)
            labels = labels.to(device=device)
            outputs = model(imgs)
            loss = loss_fn(outputs, labels)

            l2_lambda = 0.001
            l2_norm = sum(p.pow(2.0).sum() for p in model.parameters())
            loss = loss +  l2_lambda * l2_norm

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_train += loss.item()

        if epoch == 1 or epoch % 10 == 0:
            print(f"Epoch {epoch}, Training loss {loss_train}")

## Dropout

Dropout to technika regularyzacji stosowana w **sieciach neuronowych**, która pomaga ograniczyć **overfitting (przeuczenie)**.  
W przeciwieństwie do regularyzacji L2 (która wpływa bezpośrednio na wartości wag), dropout działa na poziomie **architektury sieci podczas treningu**.

---

### 🧠 Na czym polega dropout?

Podczas treningu losowo „wyłączamy” (zerujemy) część neuronów w sieci.

- dla każdej iteracji losowana jest inna „maska” neuronów  
- każdy neuron ma określone prawdopodobieństwo wyłączenia \(p\)  
- sieć nigdy nie widzi danych w pełnej konfiguracji  

Efekt: model nie może polegać na konkretnych neuronach → musi uczyć się bardziej rozproszonych i ogólnych reprezentacji.

---

### 🔧 Intuicja

Dropout można interpretować jako:
- trenowanie wielu różnych „podsieci” jednocześnie  
- wymuszanie redundancji w reprezentacjach  
- ograniczenie „współuzależnienia” neuronów (co-adaptation)

**Uwaga**:  
Dropout jest przykładem mechanizmu, który zachowuje się inaczej w trakcie treningu i walidacji. Tworząc modele w PyTorch należy pamiętać, aby przełączyć je w odpowiedni tryb: `model.train()` lub `model.eval()`. Dobrze jest wyrobić sobie nawyk używania tych trybów nawet, jeśli w modelu nie ma dropoutu (ani np. batch normalization - patrz poniżej).

---

### ⚙️ Implementacja w PyTorch
Dla danej warstwy z dropoutem określa się prawdopodobieństwo wyłączenia (w PyTorch; lub pozostawienia - uwaga na różne konwencje!) danego neuronu.

- https://pytorch.org/docs/stable/generated/torch.nn.Dropout2d.html

```python
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(p=0.5),  # 50% neuronów wyłączanych podczas treningu
    nn.Linear(256, 10)
)

In [19]:
class NetDropout(nn.Module):
    def __init__(self, n_chans1=32):
        super().__init__()
        self.n_chans1 = n_chans1
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)
        self.conv1_dropout = nn.Dropout2d(p=0.4)
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3, padding=1)
        self.conv2_dropout = nn.Dropout2d(p=0.4)
        self.fc1 = nn.Linear(8 * 8 * n_chans1 // 2, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = F.max_pool2d(torch.tanh(self.conv1(x)), 2)
        out = self.conv1_dropout(out)
        out = F.max_pool2d(torch.tanh(self.conv2(out)), 2)
        out = self.conv2_dropout(out)
        out = out.view(-1, 8 * 8 * self.n_chans1 // 2)
        out = torch.tanh(self.fc1(out))
        out = self.fc2(out)
        return out

## Batch Normalization

Batch Normalization (BN) to technika stosowana w sieciach neuronowych w celu **stabilizacji i przyspieszenia treningu**.  
Polega na normalizacji wartości funkcji aktywacji w trakcie uczenia, dzięki czemu wartości przekazywane między warstwami mają bardziej kontrolowany rozkład.

---

### 🧠 Na czym polega Batch Normalization?

Dla każdej warstwy obliczane są:

- średnia $\mu$  
- wariancja $\sigma^2$  

a następnie dane są standaryzowane:

$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}
$$

gdzie:
- $x$ – wejście do warstwy  
- $\mu$ – średnia  
- $\sigma^2$ – wariancja  
- $\epsilon$ – mała stała zapobiegająca dzieleniu przez zero  

Po normalizacji stosowane są dodatkowe parametry uczone podczas treningu:
- skala $\gamma$
- przesunięcie $\beta$

Dzięki temu model może nauczyć się optymalnego zakresu dla wartości funkcji aktywacji.

**Uwaga**:  
Batch Normalization działa inaczej w zależności od trybu modelu:

#### trening (`model.train()`)
- średnia i wariancja liczone są na podstawie aktualnego batcha  

#### ewaluacja (`model.eval()`)
- używane są uśrednione statystyki zebrane podczas treningu  

---

### 🔧 Implementacja w PyTorch

Dla sieci konwolucyjnych:

```python
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU()
)

In [20]:
class NetBatchNorm(nn.Module):
    def __init__(self, n_chans1=32):
        super().__init__()
        self.n_chans1 = n_chans1
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)
        self.conv1_batchnorm = nn.BatchNorm2d(num_features=n_chans1)
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3, padding=1)
        self.conv2_batchnorm = nn.BatchNorm2d(num_features=n_chans1 // 2)
        self.fc1 = nn.Linear(8 * 8 * n_chans1 // 2, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = self.conv1_batchnorm(self.conv1(x))
        out = F.max_pool2d(torch.tanh(out), 2)
        out = self.conv2_batchnorm(self.conv2(out))
        out = F.max_pool2d(torch.tanh(out), 2)
        out = out.view(-1, 8 * 8 * self.n_chans1 // 2)
        out = torch.tanh(self.fc1(out))
        out = self.fc2(out)
        return out

## ✅ Zadanie 5

Zaimplementuj mechanizm **early stopping** w pętli treningowej.

In [21]:
def training_loop(
    n_epochs,
    optimizer,
    model,
    loss_fn,
    train_loader,
    val_loader,
    device,
    patience=10
):

    print(f"Training on: {device}")

    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(1, n_epochs + 1):

        # ===== TRAIN =====
        model.train()
        train_loss = 0.0

        for imgs, labels in train_loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)
            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # ===== VALIDATION =====
        model.eval()
        val_loss = 0.0

        with torch.no_grad():

            for imgs, labels in val_loader:

                imgs = imgs.to(device)
                labels = labels.to(device)

                outputs = model(imgs)
                loss = loss_fn(outputs, labels)

                val_loss += loss.item()

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Epoch {epoch}, "
                f"Train loss: {train_loss:.4f}, "
                f"Val loss: {val_loss:.4f}"
            )

        # ===== EARLY STOPPING =====
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0

        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

In [22]:
training_loop(
    n_epochs=100,
    optimizer=optimizer,
    model=model,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    patience=10
)

Training on: cuda
Epoch 1, Train loss: 43.6888, Val loss: 10.5321
Epoch 10, Train loss: 44.2574, Val loss: 10.0297
Early stopping at epoch 12


## ✅ Zadanie 6

Wytrenuj i przeprowadź ewalucję rozbudowanych modeli (`NetWidth`, `NetDepth`).

**Podsumuj wyniki w tabeli, wyciągnij wnioski.**

In [23]:
# NetWidth
model_width = NetWidth(filters=32).to(device)

optimizer_width = torch.optim.Adam(
    model_width.parameters(),
    lr=0.01
)

training_loop(
    n_epochs=100,
    optimizer=optimizer_width,
    model=model_width,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device
)

Training on: cuda
Epoch 1, Train loss: 74.9461, Val loss: 13.2154
Epoch 10, Train loss: 61.6500, Val loss: 11.9412
Epoch 20, Train loss: 54.5449, Val loss: 11.6003
Epoch 30, Train loss: 53.7698, Val loss: 11.0367
Early stopping at epoch 39


In [24]:
# NetDepth
model_depth = NetDepth().to(device)

optimizer_depth = torch.optim.Adam(
    model_depth.parameters(),
    lr=0.01
)

training_loop(
    n_epochs=100,
    optimizer=optimizer_depth,
    model=model_depth,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device
)

Training on: cuda
Epoch 1, Train loss: 70.6592, Val loss: 12.4603
Epoch 10, Train loss: 48.9264, Val loss: 11.7900
Epoch 20, Train loss: 45.5509, Val loss: 10.4005
Early stopping at epoch 25


# WNIOSKI

| Model        | Epoch 1 (Train / Val loss) | Epoch 10 (Train / Val loss) | Epoch 20 (Train / Val loss) | Epoch 30 (Train / Val loss) | Early Stopping |
| ------------ | -------------------------- | --------------------------- | --------------------------- | --------------------------- | -------------- |
| **Net**      | 74.9461 / 13.2154          | 61.6500 / 11.9412           | 54.5449 / 11.6003           | 53.7698 / 11.0367           | **39**         |
| **NetDepth** | 70.6592 / 12.4603          | 48.9264 / 11.7900           | 45.5509 / 10.4005           | —                           | **25**         |


Model NetDepth osiągnął niższe wartości train loss i validation loss już na wcześniejszych etapach treningu oraz zakończył uczenie szybciej (25 epoka) niż model podstawowy (39 epoka). Wartość validation loss dla modelu NetDepth była niższa (10.4005) niż dla modelu bazowego (11.0367), co wskazuje na lepsze dopasowanie modelu do danych walidacyjnych.

## ✅ Zadanie 7

W tym zadaniu sprawdzisz w praktyce, jak różne metody regularyzacji wpływają na jakość modelu uczenia głębokiego.

---

### 🎯 Cel zadania

Wybierz jeden z modeli:
- `NetWidth`
- `NetDepth`

Następnie przeprowadź trening i ewaluację modelu z użyciem różnych technik regularyzacji.

---

### 🧪 Etapy eksperymentu

Dla wybranego modelu wykonaj następujące eksperymenty:

#### 1. Model bazowy (bez regularyzacji)
- brak L2
- brak dropout
- brak batch normalization
- brak early stopping

---

#### 2. Regularyzacja pojedyncza
Przetestuj osobno każdą z metod:

- L2 (weight decay)
- Dropout
- Batch Normalization

Dla każdej metody:
- trenuj model od nowa
- zmierz metryki (dokładność, precyzja, czułość, F1) na zbiorze walidacyjnym/testowym

---

#### 3. Kombinacja metod
Stwórz model zawierający wszystkie metody jednocześnie:

- L2 (weight decay)
- Dropout
- Batch Normalization
- Early Stopping

---

### 📊 Wyniki

Dla każdego eksperymentu przygotuj tabelę zawierającą:

| Model / metoda     | Metryka train | Loss train | Metryka val | Loss val |
|--------------------|--------------|------------|-------------|----------|
| Model bazowy       |              |            |             |          |
| L2                 |              |            |             |          |
| Dropout            |              |            |             |          |
| BatchNorm          |              |            |             |          |
| Kombinacja metod   |              |            |             |          |
---

### 📌 Wnioski (część opisowa)

Odpowiedz na pytania:

- Która metoda regularyzacji działała najlepiej?
- Czy kombinacja metod poprawiła wyniki względem pojedynczych technik?
- Czy któraś metoda pogorszyła wynik?
- Jak zmienia się przeuczenie (overfitting) między eksperymentami?

---

### 💡 Wskazówki

- wszystkie modele trenuj przez tę samą liczbę epok (jeśli nie używasz early stopping)
- używaj tego samego podziału danych
- porównuj wyniki tylko przy zachowaniu tych samych warunków treningu
- zapisuj najlepszy model (early stopping)

In [25]:
import torch
import torch.nn as nn


class NetDepth(nn.Module):

    def __init__(
        self,
        use_dropout=False,
        use_batchnorm=False
    ):
        super().__init__()

        self.use_dropout = use_dropout
        self.use_batchnorm = use_batchnorm

        # conv1
        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        # conv2
        self.conv2 = nn.Conv2d(
            in_channels=16,
            out_channels=8,
            kernel_size=3,
            padding=1
        )

        # conv3 (NetDepth)
        self.conv3 = nn.Conv2d(
            in_channels=8,
            out_channels=8,
            kernel_size=3,
            padding=1
        )

        # batch norm
        self.bn1 = nn.BatchNorm2d(16)
        self.bn2 = nn.BatchNorm2d(8)
        self.bn3 = nn.BatchNorm2d(8)

        # aktywacje
        self.act1 = nn.Tanh()
        self.act2 = nn.Tanh()
        self.act3 = nn.Tanh()
        self.act4 = nn.Tanh()

        # pooling
        self.pool1 = nn.MaxPool2d(2)
        self.pool2 = nn.MaxPool2d(2)
        self.pool3 = nn.MaxPool2d(2)

        # dropout
        self.dropout = nn.Dropout(0.3)

        # linear
        self.fc1 = nn.Linear(8 * 4 * 4, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):

        x = self.conv1(x)

        if self.use_batchnorm:
            x = self.bn1(x)

        x = self.act1(x)
        x = self.pool1(x)

        x = self.conv2(x)

        if self.use_batchnorm:
            x = self.bn2(x)

        x = self.act2(x)
        x = self.pool2(x)

        x = self.conv3(x)

        if self.use_batchnorm:
            x = self.bn3(x)

        x = self.act3(x)
        x = self.pool3(x)

        x = torch.flatten(x, 1)

        if self.use_dropout:
            x = self.dropout(x)

        x = self.fc1(x)
        x = self.act4(x)
        x = self.fc2(x)

        return x


model = NetDepth().to(device)

print(model)

NetDepth(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): Tanh()
  (act2): Tanh()
  (act3): Tanh()
  (act4): Tanh()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=128, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=2, bias=True)
)


In [26]:
def training_loop(
    n_epochs,
    optimizer,
    model,
    loss_fn,
    train_loader,
    val_loader,
    device,
    early_stopping=False,
    patience=10
):

    print(f"Training on: {device}")

    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(1, n_epochs + 1):

        # train
        model.train()

        train_loss = 0.0

        for imgs, labels in train_loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)

            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # validation
        model.eval()

        val_loss = 0.0

        with torch.no_grad():

            for imgs, labels in val_loader:

                imgs = imgs.to(device)
                labels = labels.to(device)

                outputs = model(imgs)

                loss = loss_fn(outputs, labels)

                val_loss += loss.item()

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Epoch {epoch}, "
                f"Train loss: {train_loss:.4f}, "
                f"Val loss: {val_loss:.4f}"
            )

        # early stopping
        if early_stopping:

            if val_loss < best_val_loss:

                best_val_loss = val_loss
                patience_counter = 0

            else:
                patience_counter += 1

            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

In [27]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


def validate(model, loader, device):

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for imgs, labels in loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)

            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [28]:
results = {}

model_base = NetDepth().to(device)

optimizer_base = torch.optim.Adam(
    model_base.parameters(),
    lr=0.01
)

loss_fn = nn.CrossEntropyLoss()

training_loop(
    n_epochs=100,
    optimizer=optimizer_base,
    model=model_base,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    early_stopping=False
)

train_metrics = validate(
    model_base,
    train_loader,
    device
)

val_metrics = validate(
    model_base,
    val_loader,
    device
)

results["Base"] = {
    "train": train_metrics,
    "val": val_metrics
}

print("train:", train_metrics)
print("validation:", val_metrics)

Training on: cuda
Epoch 1, Train loss: 67.4523, Val loss: 12.7309
Epoch 10, Train loss: 46.7935, Val loss: 10.6128
Epoch 20, Train loss: 44.7959, Val loss: 11.0315
Epoch 30, Train loss: 45.7623, Val loss: 10.6653
Epoch 40, Train loss: 45.6194, Val loss: 10.2832
Epoch 50, Train loss: 43.9772, Val loss: 10.0727
Epoch 60, Train loss: 44.9311, Val loss: 11.2811
Epoch 70, Train loss: 42.9899, Val loss: 10.6348
Epoch 80, Train loss: 46.5751, Val loss: 12.1776
Epoch 90, Train loss: 46.5417, Val loss: 10.1974
Epoch 100, Train loss: 48.4497, Val loss: 10.6180
train: {'accuracy': 0.8747, 'precision': 0.8804060913705584, 'recall': 0.8672, 'f1': 0.8737531486146096}
validation: {'accuracy': 0.867, 'precision': 0.8752556237218814, 'recall': 0.856, 'f1': 0.865520728008089}


In [29]:
model_l2 = NetDepth().to(device)

optimizer_l2 = torch.optim.Adam(
    model_l2.parameters(),
    lr=0.01,
    weight_decay=1e-4
)

training_loop(
    n_epochs=100,
    optimizer=optimizer_l2,
    model=model_l2,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    early_stopping=False
)

train_metrics = validate(
    model_l2,
    train_loader,
    device
)

val_metrics = validate(
    model_l2,
    val_loader,
    device
)

results["L2"] = {
    "train": train_metrics,
    "val": val_metrics
}

print("train:", train_metrics)
print("validation:", val_metrics)

Training on: cuda
Epoch 1, Train loss: 67.8401, Val loss: 10.9827
Epoch 10, Train loss: 46.7551, Val loss: 9.6512
Epoch 20, Train loss: 41.8391, Val loss: 8.9536
Epoch 30, Train loss: 39.3838, Val loss: 10.2906
Epoch 40, Train loss: 39.7115, Val loss: 9.1264
Epoch 50, Train loss: 38.8686, Val loss: 9.8178
Epoch 60, Train loss: 36.9816, Val loss: 9.6778
Epoch 70, Train loss: 35.7382, Val loss: 9.4005
Epoch 80, Train loss: 35.0657, Val loss: 9.8019
Epoch 90, Train loss: 34.5533, Val loss: 9.6882
Epoch 100, Train loss: 32.5290, Val loss: 9.6648
train: {'accuracy': 0.9143, 'precision': 0.9633191679713711, 'recall': 0.8614, 'f1': 0.9095132509766656}
validation: {'accuracy': 0.8895, 'precision': 0.9401129943502825, 'recall': 0.832, 'f1': 0.8827586206896552}


In [30]:
model_dropout = NetDepth(
    use_dropout=True
).to(device)

optimizer_dropout = torch.optim.Adam(
    model_dropout.parameters(),
    lr=0.01
)

training_loop(
    n_epochs=100,
    optimizer=optimizer_dropout,
    model=model_dropout,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    early_stopping=False
)

train_metrics = validate(
    model_dropout,
    train_loader,
    device
)

val_metrics = validate(
    model_dropout,
    val_loader,
    device
)

results["Dropout"] = {
    "train": train_metrics,
    "val": val_metrics
}

print("train:", train_metrics)
print("validation:", val_metrics)

Training on: cuda
Epoch 1, Train loss: 71.2924, Val loss: 12.1932
Epoch 10, Train loss: 51.5951, Val loss: 12.1832
Epoch 20, Train loss: 51.4694, Val loss: 10.3039
Epoch 30, Train loss: 55.2565, Val loss: 10.5063
Epoch 40, Train loss: 52.2216, Val loss: 10.9752
Epoch 50, Train loss: 50.5136, Val loss: 10.9193
Epoch 60, Train loss: 51.9092, Val loss: 10.0372
Epoch 70, Train loss: 52.6518, Val loss: 10.7645
Epoch 80, Train loss: 54.0988, Val loss: 10.8944
Epoch 90, Train loss: 53.5757, Val loss: 10.4683
Epoch 100, Train loss: 53.3693, Val loss: 10.0801
train: {'accuracy': 0.8667, 'precision': 0.8486404259364898, 'recall': 0.8926, 'f1': 0.8700653085096013}
validation: {'accuracy': 0.8615, 'precision': 0.8499515972894482, 'recall': 0.878, 'f1': 0.8637481554353172}


In [31]:
model_bn = NetDepth(
    use_batchnorm=True
).to(device)

optimizer_bn = torch.optim.Adam(
    model_bn.parameters(),
    lr=0.01
)

training_loop(
    n_epochs=100,
    optimizer=optimizer_bn,
    model=model_bn,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    early_stopping=False
)

train_metrics = validate(
    model_bn,
    train_loader,
    device
)

val_metrics = validate(
    model_bn,
    val_loader,
    device
)

results["BatchNorm"] = {
    "train": train_metrics,
    "val": val_metrics
}

print("train:", train_metrics)
print("validation:", val_metrics)

Training on: cuda
Epoch 1, Train loss: 67.1294, Val loss: 11.4109
Epoch 10, Train loss: 34.7503, Val loss: 9.2413
Epoch 20, Train loss: 21.1793, Val loss: 12.4137
Epoch 30, Train loss: 13.9007, Val loss: 12.7437
Epoch 40, Train loss: 9.9615, Val loss: 15.4278
Epoch 50, Train loss: 9.6993, Val loss: 15.8644
Epoch 60, Train loss: 5.6892, Val loss: 16.7314
Epoch 70, Train loss: 6.3919, Val loss: 18.1565
Epoch 80, Train loss: 4.8659, Val loss: 21.2294
Epoch 90, Train loss: 6.1435, Val loss: 20.4774
Epoch 100, Train loss: 3.5172, Val loss: 23.8225
train: {'accuracy': 0.9896, 'precision': 0.996149169031212, 'recall': 0.983, 'f1': 0.9895309039661767}
validation: {'accuracy': 0.881, 'precision': 0.9018987341772152, 'recall': 0.855, 'f1': 0.87782340862423}


In [32]:
model_combo = NetDepth(
    use_dropout=True,
    use_batchnorm=True
).to(device)

optimizer_combo = torch.optim.Adam(
    model_combo.parameters(),
    lr=0.01,
    weight_decay=1e-4
)

training_loop(
    n_epochs=100,
    optimizer=optimizer_combo,
    model=model_combo,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    early_stopping=True,
    patience=10
)

train_metrics = validate(
    model_combo,
    train_loader,
    device
)

val_metrics = validate(
    model_combo,
    val_loader,
    device
)

results["Combination"] = {
    "train": train_metrics,
    "val": val_metrics
}

print("train:", train_metrics)
print("validation:", val_metrics)

Training on: cuda
Epoch 1, Train loss: 70.8414, Val loss: 11.7301
Epoch 10, Train loss: 42.6937, Val loss: 9.0802
Epoch 20, Train loss: 37.4950, Val loss: 7.6429
Epoch 30, Train loss: 33.7620, Val loss: 7.6540
Early stopping at epoch 37
train: {'accuracy': 0.9204, 'precision': 0.8894034827713968, 'recall': 0.9602, 'f1': 0.9234468166955184}
validation: {'accuracy': 0.886, 'precision': 0.8567467652495379, 'recall': 0.927, 'f1': 0.8904899135446686}


In [33]:
import pandas as pd

table = []

for name, metrics in results.items():

    table.append({
        "Model / metoda": name,

        "Train accuracy":
            round(metrics["train"]["accuracy"], 4),

        "Train precision":
            round(metrics["train"]["precision"], 4),

        "Train recall":
            round(metrics["train"]["recall"], 4),

        "Train F1":
            round(metrics["train"]["f1"], 4),

        "Validation accuracy":
            round(metrics["val"]["accuracy"], 4),

        "Validation precision":
            round(metrics["val"]["precision"], 4),

        "Validation recall":
            round(metrics["val"]["recall"], 4),

        "Validation F1":
            round(metrics["val"]["f1"], 4)
    })

results_df = pd.DataFrame(table)

results_df

,Model / metoda,Train accuracy,Train precision,Train recall,Train F1,Validation accuracy,Validation precision,Validation recall,Validation F1
0,Base,0.8747,0.8804,0.8672,0.8738,0.8670,0.8753,0.856,0.8655
1,L2,0.9143,0.9633,0.8614,0.9095,0.8895,0.9401,0.832,0.8828
2,Dropout,0.8667,0.8486,0.8926,0.8701,0.8615,0.8500,0.878,0.8637
3,BatchNorm,0.9896,0.9961,0.9830,0.9895,0.8810,0.9019,0.855,0.8778
4,Combination,0.9204,0.8894,0.9602,0.9234,0.8860,0.8567,0.927,0.8905


| Model / metoda   | Metryka train                                     | Loss train | Metryka val                                       | Loss val |
| ---------------- | ------------------------------------------------- | ---------: | ------------------------------------------------- | -------: |
| Model bazowy     | Acc=0.8747, Prec=0.8804, Recall=0.8672, F1=0.8738 |    48.4497 | Acc=0.8670, Prec=0.8753, Recall=0.8560, F1=0.8655 |  10.6180 |
| L2               | Acc=0.9143, Prec=0.9633, Recall=0.8614, F1=0.9095 |    32.5290 | Acc=0.8895, Prec=0.9401, Recall=0.8320, F1=0.8828 |   9.6648 |
| Dropout          | Acc=0.8667, Prec=0.8486, Recall=0.8926, F1=0.8701 |    53.3693 | Acc=0.8615, Prec=0.8500, Recall=0.8780, F1=0.8637 |  10.0801 |
| BatchNorm        | Acc=0.9896, Prec=0.9961, Recall=0.9830, F1=0.9895 |     3.5172 | Acc=0.8810, Prec=0.9019, Recall=0.8550, F1=0.8778 |  23.8225 |
| Kombinacja metod | Acc=0.9204, Prec=0.8894, Recall=0.9602, F1=0.9234 |    33.7620 | Acc=0.8860, Prec=0.8567, Recall=0.9270, F1=0.8905 |   7.6540 |


# Odpowiedzi do zadania

## Która metoda regularyzacji działała najlepiej?

Najlepsze wyniki uzyskała kombinacja metod regularyzacji, osiągając najwyższy wynik walidacyjny (accuracy = 0.8860, F1 = 0.8905). Spośród pojedynczych metod najlepiej wypadło L2, natomiast BatchNorm osiągnął bardzo wysokie wyniki na zbiorze treningowym, ale wykazywał oznaki przeuczenia.

## Czy kombinacja metod poprawiła wyniki względem pojedynczych technik?
Tak, połączenie L2, Dropout, BatchNorm oraz Early Stopping poprawiło jakość klasyfikacji względem większości pojedynczych metod. Model osiągnął bardziej stabilne wyniki na zbiorze walidacyjnym i lepszą zdolność generalizacji.
## Czy któraś metoda pogorszyła wynik?
Tak, Dropout zastosowany samodzielnie dał najniższe wyniki walidacyjne (accuracy = 0.8615, F1 = 0.8637), co oznacza niewielkie pogorszenie względem modelu bazowego.

## Jak zmienia się przeuczenie (overfitting) między eksperymentami?
Największe przeuczenie wystąpiło przy BatchNorm, gdzie metryki treningowe były bardzo wysokie, a walidacyjne wyraźnie niższe. L2 oraz kombinacja metod ograniczyły overfitting, dając bardziej zbliżone wyniki dla zbioru treningowego i walidacyjnego.

# WNIOSKI
Podczas laboratorium nauczyłem się budowy i trenowania splotowych sieci neuronowych CNN w środowisku PyTorch. Zrozumiałem działanie metod regularyzacji, takich jak L2, Dropout, Batch Normalization oraz Early Stopping i ich wpływ na skuteczność modelu. Nauczyłem się analizować metryki jakości, takie jak accuracy, precision, recall i F1-score oraz porównywać wyniki różnych architektur. Zobaczyłem również, jak odpowiednio dobrane techniki regularyzacji pomagają ograniczyć przeuczenie modelu.